In [ ]:
import ast
import html
import math
import operator as op
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

_ALLOWED_BINOPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
}
_ALLOWED_UNARY = {
    ast.UAdd: op.pos,
    ast.USub: op.neg,
}
_ALLOWED_NAMES = {
    "pi": math.pi,
    "e": math.e,
}

def _safe_number(text):
    """Convierte una expresión matemática sencilla en un número."""
    if text is None:
        return None
    text = str(text).strip().replace(",", ".").replace("π", "pi").replace("^", "**")
    if not text:
        return None

    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return float(node.value)
        if isinstance(node, ast.Name) and node.id in _ALLOWED_NAMES:
            return float(_ALLOWED_NAMES[node.id])
        if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINOPS:
            return _ALLOWED_BINOPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARY:
            return _ALLOWED_UNARY[type(node.op)](_eval(node.operand))
        raise ValueError("Expresión no válida")

    try:
        return float(_eval(ast.parse(text, mode="eval")))
    except Exception:
        return None


def numeric_group(var_name, labels):
    """Grupo de respuestas numéricas almacenadas en una lista."""
    answers = [None] * len(labels)
    globals()[var_name] = answers
    controls = []
    status = widgets.HTML(
        value="<span style='color:#666'>Introduce las respuestas. Se registran automáticamente.</span>"
    )

    def make_handler(index):
        def _handler(change):
            answers[index] = _safe_number(change["new"])
            globals()[var_name] = list(answers)
            if change["new"].strip() and answers[index] is None:
                status.value = "<span style='color:#b00020'>Hay una expresión numérica no válida.</span>"
            else:
                status.value = "<span style='color:#2e7d32'>Respuesta registrada.</span>"
        return _handler

    for i, label in enumerate(labels):
        w = widgets.Text(
            value="",
            placeholder="Escribe un valor",
            description=label,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="420px"),
        )
        w.observe(make_handler(i), names="value")
        controls.append(w)

    display(widgets.VBox(controls + [status]))
    return controls


def radio_group(var_name, items, options):
    """Una respuesta de opción única para cada elemento."""
    answers = [""] * len(items)
    globals()[var_name] = answers
    boxes = []
    full_options = [("— Selecciona —", "")] + list(options)

    def make_handler(index):
        def _handler(change):
            answers[index] = change["new"]
            globals()[var_name] = list(answers)
        return _handler

    for i, item in enumerate(items):
        title = widgets.HTMLMath(value=f"<b>{item}</b>")
        rb = widgets.RadioButtons(
            options=full_options,
            value="",
            layout=widgets.Layout(width="95%"),
        )
        rb.observe(make_handler(i), names="value")
        boxes.append(widgets.VBox([title, rb]))

    display(widgets.VBox(boxes))
    return boxes


def checkbox_group(var_name, items, options):
    """Selección múltiple mediante casillas para cada elemento."""
    answers = [[] for _ in items]
    globals()[var_name] = answers
    groups = []

    def refresh(group_index, checkboxes):
        selected = [
            code for checkbox, (_, code) in zip(checkboxes, options)
            if checkbox.value
        ]
        answers[group_index] = selected
        globals()[var_name] = [list(x) for x in answers]

    for i, item in enumerate(items):
        title = widgets.HTMLMath(value=f"<b>{item}</b>")
        checks = [widgets.Checkbox(value=False, description=label) for label, _ in options]
        for cb in checks:
            cb.observe(lambda change, i=i, checks=checks: refresh(i, checks), names="value")
        groups.append(widgets.VBox([title] + checks))

    display(widgets.VBox(groups))
    return groups


def mixed_superposition_widget(var_name):
    """Widget específico para el ejercicio de superposición."""
    answers = [None, None, None, ""]
    globals()[var_name] = answers

    texts = [
        widgets.Text(description="a)", placeholder="valor", style={"description_width":"initial"}),
        widgets.Text(description="b)", placeholder="valor", style={"description_width":"initial"}),
        widgets.Text(description="c)", placeholder="valor", style={"description_width":"initial"}),
    ]
    yesno = widgets.RadioButtons(
        options=[("— Selecciona —",""), ("Sí","SI"), ("No","NO")],
        value="",
        description="d)",
        style={"description_width":"initial"},
    )

    for i, w in enumerate(texts):
        def handler(change, i=i):
            answers[i] = _safe_number(change["new"])
            globals()[var_name] = list(answers)
        w.observe(handler, names="value")

    def handler_yesno(change):
        answers[3] = change["new"]
        globals()[var_name] = list(answers)

    yesno.observe(handler_yesno, names="value")
    display(widgets.VBox(texts + [yesno]))
    return texts, yesno


def manual_text_widget(var_name, placeholder="Escribe aquí tu razonamiento..."):
    """Respuesta abierta; el botón deja una copia visible para la exportación a PDF."""
    globals()[var_name] = ""
    text = widgets.Textarea(
        value="",
        placeholder=placeholder,
        layout=widgets.Layout(width="95%", height="180px"),
    )
    save = widgets.Button(description="Guardar respuesta", button_style="primary")
    out = widgets.Output()

    def _update(change):
        globals()[var_name] = change["new"]

    def _save(_):
        globals()[var_name] = text.value
        with out:
            clear_output()
            rendered = html.escape(text.value).replace("\n", "<br>")
            display(HTML(
                "<div style='border:1px solid #bbb;padding:10px;border-radius:6px'>"
                "<b>Respuesta guardada:</b><br>" + rendered + "</div>"
            ))

    text.observe(_update, names="value")
    save.on_click(_save)
    display(widgets.VBox([text, save, out]))
    return text

# Hoja de ejercicios — Señales y sistemas

## Objetivos

Esta hoja está diseñada para trabajar los conceptos fundamentales del **Tema 1** sin necesidad de programar.  
Las respuestas se introducen mediante **widgets de Jupyter** y los ejercicios autocorregibles se comprueban con **Otter Grader**.

### Cómo utilizar la hoja

1. Ejecuta primero la celda de configuración que aparece a continuación.
2. Responde utilizando los cuadros numéricos, botones de opción o casillas de selección.
3. Después de responder cada ejercicio, ejecuta la celda `grader.check("qX")` que Otter añade automáticamente a la versión del estudiante.
4. Si cambias una respuesta, vuelve a ejecutar `grader.check("qX")`.

> **Importante:** el notebook está configurado para guardar el entorno en el registro de Otter. Por ello, debes ejecutar el `grader.check` correspondiente **después de dejar tu respuesta definitiva**.

En las respuestas numéricas puedes escribir decimales o expresiones sencillas como:

- `0.5`
- `1/6`
- `3*pi/5`
- `40*pi`

También se acepta la coma decimal.

Las preguntas **11 y 12** son de respuesta abierta y se corrigen manualmente.

## 1. Características de una señal discreta

Considere

$$
x[n]=
\begin{cases}
2,&n=-3\\
-1,&n=-2\\
3,&n=-1\\
4,&n=0\\
2,&n=1\\
-2,&n=2\\
1,&n=3\\
0,&\text{en otro caso}.
\end{cases}
$$

Determine:

**a)** $ x[-2]$   
**b)** $ x[0]$   
**c)** el valor máximo de la señal  
**d)** el instante $ n$  en el que se alcanza el máximo  
**e)** el número de muestras no nulas.

In [2]:
numeric_group("q1", ["a) x[-2] =", "b) x[0] =", "c) máximo =", "d) n del máximo =", "e) nº no nulas ="]);

In [ ]:
# BEGIN SOLUTION
q1 = [-1, 4, 4, 0, 7]
# END SOLUTION

In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q1(q1, np):
    assert q1 is not None
    assert len(q1) == 5
    assert all(v is not None for v in q1)
    assert np.allclose(q1, [-1, 4, 4, 0, 7])

test_q1(q1, np)

## 2. Desplazamiento temporal

Sea

$$
x[0]=1,\qquad x[1]=2,\qquad x[2]=4,\qquad x[3]=3,
$$

con $ x[n]=0$  en otro caso, y defina

$$
y[n]=x[n-3].
$$

Determine:

**a)** $ y[3]$   
**b)** $ y[4]$   
**c)** $ y[5]$   
**d)** el instante donde $ y[n]$  alcanza su máximo  
**e)** cuántas muestras se ha desplazado la señal hacia la derecha.

In [5]:
numeric_group("q2", ["a) y[3] =", "b) y[4] =", "c) y[5] =", "d) n del máximo =", "e) desplazamiento ="]);

In [ ]:
q2 = [1, 2, 4, 5, 3] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q2(q2, np):
    assert all(v is not None for v in q2)
    assert np.allclose(q2, [1, 2, 4, 5, 3]);

test_q2(q2, np)

## 3. Inversión y desplazamiento

Utilice la misma señal $ x[n]$  del ejercicio anterior y defina

$$
y[n]=x[2-n].
$$

Determine:

**a)** $ y[-1]$   
**b)** $ y[0]$   
**c)** $ y[1]$   
**d)** $ y[2]$   
**e)** el instante $ n$  donde $ y[n]$  alcanza su máximo.

In [8]:
numeric_group("q3", ["a) y[-1] =", "b) y[0] =", "c) y[1] =", "d) y[2] =", "e) n del máximo ="]);

In [ ]:

q3 = [3, 4, 2, 1, 0] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q3(q3, np):
    assert all(v is not None for v in q3)
    assert np.allclose(q3, [3, 4, 2, 1, 0]);

test_q3(q3, np)

## 4. Parte par y parte impar

Sea

$$
x[-2]=-1,\quad x[-1]=2,\quad x[0]=4,\quad x[1]=6,\quad x[2]=3.
$$

Utilizando

$$
x_p[n]=\frac{x[n]+x[-n]}{2},\qquad
x_i[n]=\frac{x[n]-x[-n]}{2},
$$

calcule:

**a)** $ x_p[0]$   
**b)** $ x_p[1]$   
**c)** $ x_p[2]$   
**d)** $ x_i[1]$   
**e)** $ x_i[2]$ .

In [11]:
numeric_group("q4", ["a) xp[0] =", "b) xp[1] =", "c) xp[2] =", "d) xi[1] =", "e) xi[2] ="]);

In [ ]:

q4 = [4, 4, 1, 2, 2] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q4(q4, np):
    assert all(v is not None for v in q4)
    assert np.allclose(q4, [4, 4, 1, 2, 2]);

test_q4(q4, np)

## 5. Periodicidad en tiempo continuo

Determine el **periodo fundamental $ T_0$ **, en segundos, de:

**a)** $ x_1(t)=\cos(4\pi t)$ 

**b)** $ x_2(t)=\sin(10\pi t+\pi/3)$ 

**c)** $ x_3(t)=\cos(2\pi t)+\sin(4\pi t)$ 

**d)** $ x_4(t)=\cos(6\pi t)+\sin(4\pi t)$ 

In [14]:
numeric_group("q5", ["a) T0 =", "b) T0 =", "c) T0 =", "d) T0 ="]);

In [ ]:

q5 = [0.5, 0.2, 1.0, 1.0] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q5(q5, np):
    assert all(v is not None for v in q5)
    assert np.allclose(q5, [0.5, 0.2, 1.0, 1.0], rtol=1e-6, atol=1e-8)

test_q5(q5, np)

## 6. Periodicidad en tiempo discreto

Determine el periodo fundamental $ N_0$ . Si la señal **no es periódica**, introduzca `0`.

**a)** $ \displaystyle x_1[n]=\cos\left(\frac{\pi}{4}n\right)$ 

**b)** $ \displaystyle x_2[n]=\cos\left(\frac{\pi}{3}n\right)$ 

**c)** $ \displaystyle x_3[n]=\cos\left(\frac{2\pi}{5}n\right)$ 

**d)** $ \displaystyle x_4[n]=\cos(n)$ 

**e)** $ \displaystyle x_5[n]=\cos\left(\frac{\pi}{2}n\right)+\sin\left(\frac{\pi}{3}n\right)$ 

In [17]:
numeric_group("q6", ["a) N0 =", "b) N0 =", "c) N0 =", "d) N0 =", "e) N0 ="]);

In [ ]:

q6 = [8, 6, 5, 0, 12] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q6(q6, np):
    assert all(v is not None for v in q6)
    assert np.allclose(q6, [8, 6, 5, 0, 12]);

test_q6(q6, np)

## 7. Energía de una señal discreta

Sea

$$
x[-2]=-2,\quad x[-1]=1,\quad x[0]=3,\quad x[1]=2,\quad x[2]=-1.
$$

Calcule:

**a)** la energía total  
**b)** la energía correspondiente a $ n<0$   
**c)** la energía correspondiente a $ n>0$   
**d)** el porcentaje de la energía total situado en $ n=0$ .

In [20]:
numeric_group("q7", ["a) E total =", "b) E para n<0 =", "c) E para n>0 =", "d) % en n=0 ="]);

In [ ]:

q7 = [19, 5, 5, 100*9/19] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q7(q7, np):
    assert all(v is not None for v in q7)
    expected = [19, 5, 5, 100*9/19]
    assert np.allclose(q7, expected, rtol=1e-3, atol=1e-3)

test_q7(q7, np)

## 8. Señales de energía y de potencia

Para cada señal seleccione su clasificación:

1. $ \displaystyle x_1(t)=e^{-2t}u(t)$ 
2. $ \displaystyle x_2(t)=3\cos(2\pi t)$ 
3. $ \displaystyle x_3[n]=1,\;0\leq n\leq10$ , y cero en otro caso.

In [23]:
radio_group(
    "q8",
    ["1) $x_1(t)$", "2) $x_2(t)$", "3) $x_3[n]$"],
    [("Señal de energía", "E"), ("Señal de potencia", "P"), ("Ninguna de las dos", "N")]
)

[VBox(children=(HTMLMath(value='<b>1) $x_1(t)$</b>'), RadioButtons(layout=Layout(width='95%'), options=(('— Selecciona —', ''), ('Señal de energía', 'E'), ('Señal de potencia', 'P'), ('Ninguna de las dos', 'N')), value=''))),

In [ ]:

q8 = ["E", "P", "E"] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q8(q8):
    assert q8 == ["E", "P", "E"]

test_q8(q8)

## 9. Propiedades de sistemas

Para cada sistema seleccione **todas las propiedades que cumple**:

- **A.** Lineal
- **B.** Invariante en el tiempo
- **C.** Causal
- **D.** Sin memoria

$$
\begin{aligned}
S_1:&\quad y(t)=2x(t)\\
S_2:&\quad y(t)=x(t)+3\\
S_3:&\quad y(t)=x^2(t)\\
S_4:&\quad y(t)=x(t-2)\\
S_5:&\quad y(t)=t\,x(t)
\end{aligned}
$$

In [26]:
checkbox_group(
    "q9",
    ["1) $y(t)=2x(t)$", "2) $y(t)=x(t)+3$", "3) $y(t)=x^2(t)$",
     "4) $y(t)=x(t-2)$", "5) $y(t)=t\,x(t)$"],
    [("A. Lineal", "A"), ("B. Invariante en el tiempo", "B"),
     ("C. Causal", "C"), ("D. Sin memoria", "D")]
)

<>:4: SyntaxWarning: invalid escape sequence '\,'
<>:4: SyntaxWarning: invalid escape sequence '\,'
C:\Users\usuario\AppData\Local\Temp\ipykernel_31032\2990485948.py:4: SyntaxWarning: invalid escape sequence '\,'
  "4) $y(t)=x(t-2)$", "5) $y(t)=t\,x(t)$"],


[VBox(children=(HTMLMath(value='<b>1) $y(t)=2x(t)$</b>'), Checkbox(value=False, description='A. Lineal'), Checkbox(value=False, description='B. Invariante en el tiempo'), Checkbox(value=False, description='C. Causal'), Checkbox(value=False, description='D. Sin memoria'))),

In [ ]:

q9 = [["A","B","C","D"], ["B","C","D"], ["B","C","D"], ["A","B","C"], ["A","C","D"]] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 3
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q9(q9):
    expected = [set("ABCD"), set("BCD"), set("BCD"), set("ABC"), set("ACD")]
    assert len(q9) == 5
    assert [set(x) for x in q9] == expected

test_q9(q9)

## 10. Principio de superposición

Sea

$$
T\{x(t)\}=3x(t).
$$

En $ t=2$ ,

$$
x_1(2)=4,\qquad x_2(2)=-1,\qquad a=2,\qquad b=3.
$$

Calcule:

**a)** $ ax_1(2)+bx_2(2)$ 

**b)** $ \left.T\{ax_1+bx_2\}\right|_{t=2}$ 

**c)** $ \left.aT\{x_1\}\right|_{t=2}+\left.bT\{x_2\}\right|_{t=2}$ 

**d)** ¿Se cumple el principio de superposición?

In [29]:
mixed_superposition_widget("q10")

([Text(value='', description='a)', placeholder='valor', style=TextStyle(description_width='initial')),
  Text(value='', description='b)', placeholder='valor', style=TextStyle(description_width='initial')),
  Text(value='', description='c)', placeholder='valor', style=TextStyle(description_width='initial'))],
 RadioButtons(description='d)', options=(('— Selecciona —', ''), ('Sí', 'SI'), ('No', 'NO')), style=DescriptionStyle(description_width='initial'), value=''))

In [ ]:

q10 = [5, 15, 15, "SI"] # SOLUTIONd


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q10(q10, np):
    assert all(v is not None for v in q10[:3]);
    assert np.allclose(q10[:3], [5, 15, 15]);
    assert q10[3] == "SI"

test_q10(q10, np)

## 11. Razonamiento sobre periodicidad — corrección manual

Considere

$$
x_1[n]=\cos\left(\frac{\pi}{4}n\right),
\qquad
x_2[n]=\cos(n).
$$

La primera señal es periódica y la segunda no.

**Explique por qué una sinusoide no es necesariamente periódica en tiempo discreto e indique la condición que debe cumplir su frecuencia angular para que sea periódica.**

In [32]:
manual_text_widget("q11")

Textarea(value='', layout=Layout(height='180px', width='95%'), placeholder='Escribe aquí tu razonamiento...')

Una sinusoide discreta

$$
x[n]=A\cos(\omega_0n+\phi)
$$

es periódica si existe un entero positivo $ N$  y un entero $ k$  tales que

$$
\omega_0N=2\pi k.
$$

Equivalentemente, $ \omega_0/(2\pi)$  debe ser un número racional.

## 12. Análisis de un sistema — corrección manual

Considere

$$
y(t)=x(2t).
$$

Determine si el sistema es:

- lineal;
- invariante en el tiempo;
- causal;
- con memoria o sin memoria.

**Justifique brevemente cada respuesta utilizando las definiciones correspondientes.**

In [33]:
manual_text_widget("q12")

Textarea(value='', layout=Layout(height='180px', width='95%'), placeholder='Escribe aquí tu razonamiento...')

El sistema es **lineal**, **no invariante en el tiempo**, **no causal** y **con memoria**.

La linealidad se conserva porque el escalado de la variable independiente no altera el principio de superposición. No es invariante en el tiempo porque desplazar la entrada y aplicar el sistema no produce, en general, el mismo resultado que aplicar el sistema y desplazar la salida. Es no causal porque para $ t>0$ , $ y(t)$  depende de $ x(2t)$ , que corresponde a un instante futuro. Finalmente, tiene memoria porque $ y(t)$  depende, en general, de un valor de la entrada distinto de $ x(t)$ .

## 13. Escalón unitario

Sea

$$
x[n]=u[n+2]-u[n-3],
$$

donde $ u[n]=1$  para $ n\geq0$  y $ u[n]=0$  para $ n<0$ .

Determine:

**a)** el primer índice donde $ x[n]\neq0$   
**b)** el último índice donde $ x[n]\neq0$   
**c)** el número de muestras no nulas  
**d)** $ x[3]$ .

In [34]:
numeric_group("q13", ["a) primer n =", "b) último n =", "c) nº no nulas =", "d) x[3] ="]);

In [ ]:

q13 = [-2, 2, 5, 0] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q13(q13, np):
    assert all(v is not None for v in q13)
    assert np.allclose(q13, [-2, 2, 5, 0]);

test_q13(q13, np)

## 14. Representación mediante impulsos

Sea

$$
x[n]=2\delta[n+1]-3\delta[n]+4\delta[n-2].
$$

Calcule:

**a)** $ x[-1]$   
**b)** $ x[0]$   
**c)** $ x[2]$   
**d)** $ \displaystyle\sum_{n=-\infty}^{\infty}x[n]$ .

In [37]:
numeric_group("q14", ["a) x[-1] =", "b) x[0] =", "c) x[2] =", "d) suma ="]);

In [ ]:

q14 = [2, -3, 4, 3] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q14(q14, np):
    assert all(v is not None for v in q14)
    assert np.allclose(q14, [2, -3, 4, 3]);

test_q14(q14, np)

## 15. Interpretación de transformaciones temporales

Identifique la transformación correspondiente en cada caso:

1. $ x(t-4)$ 
2. $ x(t+2)$ 
3. $ x(-t)$ 
4. $ x(3t)$ 
5. $ x(t/2)$ 

In [40]:
radio_group(
    "q15",
    ["1) $x(t-4)$", "2) $x(t+2)$", "3) $x(-t)$", "4) $x(3t)$", "5) $x(t/2)$"],
    [
        ("Desplazamiento 4 unidades a la derecha", "A"),
        ("Desplazamiento 2 unidades a la izquierda", "B"),
        ("Inversión temporal", "C"),
        ("Compresión temporal por factor 3", "D"),
        ("Expansión temporal por factor 2", "E"),
    ]
);

In [ ]:

q15 = ["A", "B", "C", "D", "E"] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q15(q15):
    assert q15 == ["A", "B", "C", "D", "E"]

test_q15(q15)

## 16. Simetría de señales

Clasifique cada señal como **par**, **impar** o **ninguna de las dos**.

1. $ x_1(t)=\cos(3t)$ 
2. $ x_2(t)=\sin(2t)$ 
3. $ x_3(t)=e^t$ 
4. $ x_4(t)=t^2+2$ 
5. $ x_5(t)=t^3+t$ 

In [43]:
radio_group(
    "q16",
    ["1) $\\cos(3t)$", "2) $\\sin(2t)$", "3) $e^t$", "4) $t^2+2$", "5) $t^3+t$"],
    [("Par", "P"), ("Impar", "I"), ("Ninguna", "N")]
);

In [ ]:

q16 = ["P", "I", "N", "P", "I"] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q16(q16):
    assert q16 == ["P", "I", "N", "P", "I"]

test_q16(q16)

## 17. Parámetros de una sinusoide continua

Para

$$
x(t)=5\cos\left(40\pi t+\frac{\pi}{6}\right),
$$

determine:

**a)** la frecuencia angular $ \omega_0$  en rad/s  
**b)** la frecuencia $ f_0$  en Hz  
**c)** el periodo fundamental $ T_0$  en segundos  
**d)** la amplitud.

In [46]:
numeric_group("q17", ["a) ω0 =", "b) f0 =", "c) T0 =", "d) amplitud ="]);

In [ ]:

q17 = [40*np.pi, 20, 0.05, 5] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q17(q17, np):
    assert all(v is not None for v in q17)
    assert np.allclose(q17, [40*np.pi, 20, 0.05, 5], rtol=1e-5, atol=1e-7)

test_q17(q17, np)

## 18. Sinusoide en tiempo discreto

Sea

$$
x[n]=3\cos\left(\frac{3\pi}{5}n+\frac{\pi}{4}\right).
$$

Determine:

**a)** la amplitud  
**b)** la frecuencia angular digital $ \omega_0$ , en rad/muestra  
**c)** el periodo fundamental $ N_0$ .

In [49]:
numeric_group("q18", ["a) amplitud =", "b) ω0 =", "c) N0 ="]);

In [ ]:

q18 = [3, 3*np.pi/5, 10] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q18(q18, np):
    assert all(v is not None for v in q18)
    assert np.allclose(q18, [3, 3*np.pi/5, 10], rtol=1e-5, atol=1e-7)

test_q18(q18, np)

## 19. Periodicidad de sumas de señales discretas

Determine el periodo fundamental $ N_0$ . Si la señal no es periódica, introduzca `0`.

**a)**

$$
x_1[n]=\cos\left(\frac{\pi}{3}n\right)+
\sin\left(\frac{\pi}{4}n\right)
$$

**b)**

$$
x_2[n]=\cos\left(\frac{2\pi}{7}n\right)+
\cos\left(\frac{4\pi}{5}n\right)
$$

**c)**

$$
x_3[n]=\cos(\sqrt{2}\,n).
$$

In [52]:
numeric_group("q19", ["a) N0 =", "b) N0 =", "c) N0 ="]);

In [ ]:

q19 = [24, 35, 0] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q19(q19, np):
    assert all(v is not None for v in q19)
    assert np.allclose(q19, [24, 35, 0])

test_q19(q19, np)

## 20. Energía de señales

Calcule la energía total de las siguientes señales:

**a)**

$$
x_1(t)=e^{-3t}u(t)
$$

**b)**

$$
x_2(t)=2e^{-2t}u(t)
$$

**c)**

$$
x_3[n]=\left(\frac13\right)^n u[n].
$$

In [55]:
numeric_group("q20", ["a) E1 =", "b) E2 =", "c) E3 ="]);

In [ ]:

q20 = [1/6, 1, 9/8] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q20(q20, np):
    assert all(v is not None for v in q20)
    assert np.allclose(q20, [1/6, 1, 9/8], rtol=1e-5, atol=1e-7)

test_q20(q20, np)

## 21. Potencia media

Calcule la potencia media de:

**a)**

$$
x_1(t)=4\cos\left(6\pi t+\frac{\pi}{4}\right)
$$

**b)**

$$
x_2(t)=2+3\cos(10\pi t)
$$

**c)**

$$
x_3[n]=(-1)^n.
$$

In [58]:
numeric_group("q21", ["a) P1 =", "b) P2 =", "c) P3 ="]);

In [ ]:

q21 = [8, 8.5, 1] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q21(q21, np):
    assert all(v is not None for v in q21)
    assert np.allclose(q21, [8, 8.5, 1], rtol=1e-5, atol=1e-7)

test_q21(q21, np)

## 22. Clasificación completa de señales

Para cada señal seleccione **todas las propiedades que cumple**:

- **A.** Periódica
- **B.** Aperiódica
- **C.** Par
- **D.** Impar
- **E.** Señal de energía
- **F.** Señal de potencia

1. $ \displaystyle x_1(t)=e^{-|t|}$ 
2. $ \displaystyle x_2(t)=\sin(2\pi t)$ 
3. $ \displaystyle x_3[n]=1$  para $ -2\le n\le2$ , y cero en otro caso
4. $ \displaystyle x_4[n]=(-1)^n$ 

In [61]:
checkbox_group(
    "q22",
    ["1) $e^{-|t|}$", "2) $\\sin(2\\pi t)$",
     "3) pulso discreto simétrico", "4) $(-1)^n$"],
    [
        ("A. Periódica", "A"),
        ("B. Aperiódica", "B"),
        ("C. Par", "C"),
        ("D. Impar", "D"),
        ("E. Señal de energía", "E"),
        ("F. Señal de potencia", "F"),
    ]
);

In [ ]:

q22 = [["B","C","E"], ["A","D","F"], ["B","C","E"], ["A","C","F"]] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q22(q22):
    expected = [set("BCE"), set("ADF"), set("BCE"), set("ACF")]
    assert len(q22) == 4
    assert [set(x) for x in q22] == expected

test_q22(q22)

## 23. Propiedades de sistemas: casos adicionales

Para cada sistema seleccione **todas las propiedades que cumple**:

- **A.** Lineal
- **B.** Invariante en el tiempo
- **C.** Causal
- **D.** Sin memoria

$$
\begin{aligned}
S_1:&\quad y(t)=x(t)+x(t-1)\\
S_2:&\quad y(t)=x(t+1)\\
S_3:&\quad y(t)=|x(t)|\\
S_4:&\quad y(t)=\int_{-\infty}^{t}x(\tau)\,d\tau
\end{aligned}
$$

In [64]:
checkbox_group(
    "q23",
    ["1) $x(t)+x(t-1)$", "2) $x(t+1)$", "3) $|x(t)|$",
     "4) $\\int_{-\\infty}^{t}x(\\tau)d\\tau$"],
    [("A. Lineal", "A"), ("B. Invariante en el tiempo", "B"),
     ("C. Causal", "C"), ("D. Sin memoria", "D")]
);

In [ ]:

q23 = [["A","B","C"], ["A","B"], ["B","C","D"], ["A","B","C"]] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q23(q23):
    expected = [set("ABC"), set("AB"), set("BCD"), set("ABC")]
    assert len(q23) == 4
    assert [set(x) for x in q23] == expected

test_q23(q23)

## 24. Transformación de una señal continua

Considere

$$
x(t)=
\begin{cases}
1,&-1\le t<0\\
3,&0\le t<1\\
-2,&1\le t\le2\\
0,&\text{en otro caso}
\end{cases}
$$

y defina

$$
y(t)=x(1-2t).
$$

Calcule:

**a)** $ y(-0.5)$   
**b)** $ y(0)$   
**c)** $ y(0.25)$   
**d)** $ y(0.75)$   
**e)** $ y(1)$   
**f)** extremo izquierdo del soporte de $ y(t)$   
**g)** extremo derecho del soporte de $ y(t)$ .

In [67]:
numeric_group("q24", ["a) y(-0.5) =", "b) y(0) =", "c) y(0.25) =", "d) y(0.75) =", "e) y(1) =", "f) soporte desde =", "g) soporte hasta ="]);

In [ ]:

q24 = [-2, -2, 3, 1, 1, -0.5, 1] # SOLUTION


In [ ]:
""" # BEGIN TEST CONFIG
points: 2
success_message: Correcto!
failure_message: Revisa tu respuesta.
""" # END TEST CONFIG
def test_q24(q24, np):
    assert all(v is not None for v in q24)
    assert np.allclose(q24, [-2, -2, 3, 1, 1, -0.5, 1], rtol=1e-6, atol=1e-8)

test_q24(q24, np)